# RootCause AI — Embed pipeline (Colab, GPU)

Upload `cleaned_input.csv`, chạy hết các cell, tải file output về, import vào Postgres bằng `\copy` (hướng dẫn ở cell cuối).

**Runtime > Change runtime type > GPU (T4)** trước khi chạy, không thì phần embed sẽ chậm như trên máy bạn.

In [ ]:
!pip install -q sentence-transformers einops torch pandas

## Đọc file — giữ nguyên logic bạn đã có, không đổi gì

In [ ]:
import csv
csv.field_size_limit(10**7)

with open('cleaned_input_fixed_new.csv', newline='', encoding='utf-8') as f:
    reader = csv.reader(f)
    header = next(reader)
    for i, row in enumerate(reader, start=1):
        if i == 71118:
            print(f"Số field: {len(row)}")
            print(row)
            break

In [ ]:
import csv
import pandas as pd

csv.field_size_limit(10**7)

with open('cleaned_input_fixed_new.csv', newline='', encoding='utf-8') as f:
    reader = csv.reader(f)
    header = next(reader)
    rows = list(reader)

df = pd.DataFrame(rows, columns=header)
print(len(df))
df.head()

## Tạo cột `summary` (schema yêu cầu NOT NULL, code gốc của bạn chưa có bước này)

**SỬA danh sách `SUMMARY_COLS` bên dưới cho khớp với cách bạn muốn ghép** — hiện để mặc định ghép các cột free-text + định danh chính.

In [ ]:
SUMMARY_COLS = ["MAKETXT", "MODELTXT", "YEARTXT", "COMPDESC", "CDESCR"]  # <-- chỉnh nếu cần

def build_summary(row):
    parts = [str(row[c]) for c in SUMMARY_COLS if str(row[c]).strip()]
    return " | ".join(parts)

df["summary"] = df.apply(build_summary, axis=1)

empty_summary = (df["summary"].str.strip() == "").sum()
print(f"{empty_summary} rows có summary rỗng — schema NOT NULL sẽ reject khi import nếu không xử lý")

# fallback tối thiểu để không vi phạm NOT NULL — thay bằng logic drop-row nếu bạn muốn loại bỏ hẳn
df.loc[df["summary"].str.strip() == "", "summary"] = "NO_DESCRIPTION_AVAILABLE"

## Lowercase tên cột để khớp `schema.sql`

In [ ]:
df = df.rename(columns=str.lower)

## Load model — dùng `cuda` (GPU Colab), KHÔNG dùng `mps` ở đây (đó là dành riêng cho Mac local)

In [ ]:
import torch
from sentence_transformers import SentenceTransformer

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

EMBED_MODEL_NAME = "nomic-ai/nomic-embed-text-v1.5"
EMBED_DIM = 768  # phải khớp vector(768) trong schema.sql

model = SentenceTransformer(EMBED_MODEL_NAME, trust_remote_code=True, device=device)

## Embed theo chunk (tránh OOM trên GPU free-tier Colab)

In [ ]:
from tqdm.auto import tqdm
import numpy as np

CHUNK_SIZE = 5000
BATCH_SIZE_EMBED = 64

def embed_batch(texts):
    # Nomic REQUIRES prefix cho document — bỏ qua sẽ làm retrieval kém đi, không báo lỗi
    prefixed = [f"search_document: {t if isinstance(t, str) else ''}" for t in texts]
    embeddings = model.encode(
        prefixed,
        batch_size=BATCH_SIZE_EMBED,
        show_progress_bar=False,
        normalize_embeddings=True,  # bắt buộc: index dùng vector_cosine_ops
    )
    return embeddings

all_embeddings = []
for start in tqdm(range(0, len(df), CHUNK_SIZE), desc="Embedding chunks"):
    chunk_texts = df["summary"].iloc[start:start + CHUNK_SIZE].tolist()
    chunk_emb = embed_batch(chunk_texts)
    all_embeddings.append(chunk_emb)

embeddings = np.vstack(all_embeddings)
print(embeddings.shape)  # (n_rows, 768) — nếu cột 2 không phải 768, model/schema đang lệch

## Format embedding thành literal pgvector (`[0.1,0.2,...]`) để COPY vào Postgres đọc được

In [ ]:
df["embedding"] = ["[" + ",".join(map(str, row)) + "]" for row in embeddings]

## Xuất CSV tương thích với `COPY` / `\copy` của Postgres

Date columns đã là `datetime64` — pandas sẽ ghi ra ISO format (`YYYY-MM-DD`), Postgres đọc thẳng được không cần convert thêm.

In [ ]:
OUTPUT_PATH = "complaints_ready_for_pg.csv"

df.to_csv(OUTPUT_PATH, index=False)
print(f"Saved: {OUTPUT_PATH} ({len(df)} rows)")

In [ ]:

files.download(OUTPUT_PATH)

## Import vào Postgres (chạy trên máy bạn, sau khi tải CSV về)

```bash
psql -h localhost -U rootcauseai -d rootcauseai -c "\copy complaints(<liệt kê đúng thứ tự cột trong CSV>) FROM 'complaints_ready_for_pg.csv' WITH (FORMAT csv, HEADER true)"
```

Lưu ý: thứ tự cột trong lệnh `\copy (...)` PHẢI khớp chính xác thứ tự cột trong file CSV (chính là thứ tự `df.columns` lúc `to_csv`). Nếu lệch thứ tự, Postgres sẽ nhét sai giá trị vào sai cột mà **không báo lỗi** nếu kiểu dữ liệu tình cờ tương thích (ví dụ 2 cột cùng là TEXT) — đây là lỗi âm thầm nguy hiểm nhất, luôn double-check bằng `head -1 complaints_ready_for_pg.csv`.